# RACE QA Pipeline - EDA and Training

## Section 1 - Environment & Data Loading

In [ ]:
import os
import re
import string
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROCESSED_DIR / 'figures'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

In [ ]:
# Optional download hook. Existing CSVs are used when present.
if not (RAW_DIR / 'train.csv').exists():
    import kagglehub
    print('Download the RACE dataset with kagglehub, then standardize to article, question, A, B, C, D, answer.')
    print('This project already expects train.csv, val.csv, and test.csv under data/raw/.')

train_df = pd.read_csv(RAW_DIR / 'train.csv')
val_df = pd.read_csv(RAW_DIR / 'val.csv')
test_df = pd.read_csv(RAW_DIR / 'test.csv')
required = ['article', 'question', 'A', 'B', 'C', 'D', 'answer']
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {sorted(missing)}')
    print(name, df.shape)

## Section 2 - EDA

In [ ]:
def save_plot(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=150)
    plt.close()

train_df['answer'].value_counts().sort_index().plot(kind='bar', title='Answer distribution')
save_plot('answer_distribution.png')

train_df['article'].astype(str).str.split().str.len().plot(kind='hist', bins=50, title='Article length')
save_plot('article_length_hist.png')

train_df['question'].astype(str).str.split().str.len().plot(kind='hist', bins=40, title='Question length')
save_plot('question_length_hist.png')

option_lengths = train_df[['A', 'B', 'C', 'D']].astype(str).applymap(lambda x: len(x.split()))
option_lengths.boxplot()
plt.title('Option length comparison')
save_plot('option_length_comparison.png')

qtype = train_df['question'].astype(str).str.extract(r'^(\w+)', expand=False).str.lower().value_counts().head(15)
qtype.plot(kind='bar', title='Question type breakdown')
save_plot('question_type_breakdown.png')

split_balance = pd.DataFrame({
    'train': train_df['answer'].value_counts(normalize=True).sort_index(),
    'val': val_df['answer'].value_counts(normalize=True).sort_index(),
    'test': test_df['answer'].value_counts(normalize=True).sort_index(),
})
split_balance.plot(kind='bar', title='Answer balance across splits')
save_plot('answer_balance_across_splits.png')

summary = pd.DataFrame({
    'article_words': train_df['article'].astype(str).str.split().str.len().describe(),
    'question_words': train_df['question'].astype(str).str.split().str.len().describe(),
})
summary

## Section 3 - Preprocessing

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from pipeline.preprocessing import clean_text, expand_df, prepare_text_columns, preprocess_and_build

train_exp, val_exp, test_exp = preprocess_and_build()
print(train_exp.shape, val_exp.shape, test_exp.shape)

## Section 4 - Model A (Supervised)

In [ ]:
from pipeline.model_a_train import main as train_model_a
train_model_a()

## Section 5 - Unsupervised & Semi-Supervised

In [ ]:
from scipy.sparse import load_npz
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation

X_train = load_npz(PROCESSED_DIR / 'X_train.npz')
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
sample_n = min(5000, X_train.shape[0])
X_sample = X_train[:sample_n]
y_sample = y_train[:sample_n]

inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init='auto').fit(X_sample)
    inertias.append(km.inertia_)
plt.plot(range(2, 8), inertias, marker='o')
plt.title('K-Means elbow')
save_plot('kmeans_elbow.png')

svd = TruncatedSVD(n_components=2, random_state=42)
coords = svd.fit_transform(X_sample)
plt.scatter(coords[:, 0], coords[:, 1], c=y_sample, s=4, alpha=0.5)
plt.title('SVD projection')
save_plot('svd_projection.png')

gmm = GaussianMixture(n_components=2, random_state=42).fit(coords)
labels = np.full(sample_n, -1)
labels[:max(1, sample_n // 10)] = y_sample[:max(1, sample_n // 10)]
lp = LabelPropagation().fit(coords, labels)
{'gmm_components': gmm.n_components, 'label_prop_classes': sorted(set(lp.transduction_.tolist()))}

## Section 6 - Question Generation

In [ ]:
from pipeline.inference import generate_question

example = train_df.iloc[0]
answer = example[example['answer']]
generate_question(example['article'], answer)[:5]

## Section 7 - Model B

In [ ]:
from pipeline.model_b_train import main as train_model_b
train_model_b()

## Section 8 - Final Evaluation

In [ ]:
from pipeline.evaluate import compute_generation_metrics, compute_metrics
from pipeline.inference import generate_distractors, get_hints, predict_answer

row = test_df.iloc[0]
options = [row[o] for o in ['A', 'B', 'C', 'D']]
pred = predict_answer(row['article'], row['question'], options)
distractors = generate_distractors(row['article'], row['question'], row[row['answer']])
hints = get_hints(row['article'], row['question'])
generation_metrics = compute_generation_metrics([' '.join(distractors)], [' '.join([row[o] for o in ['A', 'B', 'C', 'D'] if o != row['answer']])])
{'predicted_answer': pred, 'distractors': distractors, 'hints': hints, 'generation_metrics': generation_metrics}

## Section 9 - Export Scripts

In [ ]:
from pathlib import Path
Path('src').mkdir(exist_ok=True)

In [ ]:
%%writefile src/preprocessing.py
"""Preprocessing utilities for the RACE QA pipeline."""

import os
import re
import string

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
DEFAULT_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DEFAULT_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DEFAULT_MODELS_DIR = os.path.join(PROJECT_ROOT, "models", "model_a", "traditional")
OPTIONS = ("A", "B", "C", "D")


def _resolve_path(path_value):
    if path_value is None:
        return None
    if os.path.isabs(str(path_value)):
        return str(path_value)
    return os.path.join(PROJECT_ROOT, str(path_value))


def clean_text(text):
    """Lowercase, remove punctuation, and collapse whitespace."""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


def prepare_text_columns(df):
    """Return a copy with cleaned article, question, and option columns."""
    result = df.copy()
    result["article_clean"] = result["article"].apply(clean_text)
    result["question_clean"] = result["question"].apply(clean_text)
    for option in OPTIONS:
        result[f"{option}_clean"] = result[option].apply(clean_text)
    return result


def expand_df(df):
    """Expand each MCQ row into four binary verification rows."""
    rows = []
    for row in df.itertuples(index=False):
        row_dict = row._asdict()
        article_clean = row_dict.get("article_clean", clean_text(row_dict["article"]))
        question_clean = row_dict.get("question_clean", clean_text(row_dict["question"]))
        answer_letter = str(row_dict["answer"]).strip()

        for option in OPTIONS:
            option_clean = row_dict.get(f"{option}_clean", clean_text(row_dict[option]))
            rows.append(
                {
                    "article": article_clean,
                    "question": question_clean,
                    "option": option_clean,
                    "option_letter": option,
                    "label": 1 if answer_letter == option else 0,
                    "combined_text": f"{article_clean} [SEP] {question_clean} [SEP] {option_clean}",
                    "article_raw": row_dict["article"],
                    "question_raw": row_dict["question"],
                    "A_raw": row_dict["A"],
                    "B_raw": row_dict["B"],
                    "C_raw": row_dict["C"],
                    "D_raw": row_dict["D"],
                    "answer": answer_letter,
                }
            )
    return pd.DataFrame(rows)


def _rowwise_cosine(article_texts, option_texts, vocabulary):
    vectorizer = CountVectorizer(binary=True, vocabulary=vocabulary)
    article_matrix = vectorizer.transform(article_texts)
    option_matrix = vectorizer.transform(option_texts)
    numerator = np.asarray(article_matrix.multiply(option_matrix).sum(axis=1)).ravel()
    article_norm = np.sqrt(np.asarray(article_matrix.multiply(article_matrix).sum(axis=1)).ravel())
    option_norm = np.sqrt(np.asarray(option_matrix.multiply(option_matrix).sum(axis=1)).ravel())
    denominator = article_norm * option_norm
    values = np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=np.float32), where=denominator != 0)
    return csr_matrix(values.reshape(-1, 1))


def _lexical_features(df_expanded):
    rows = []
    for article, question, option in zip(df_expanded["article"], df_expanded["question"], df_expanded["option"]):
        article = str(article)
        question = str(question)
        option = str(option)
        article_tokens = set(article.split())
        question_tokens = set(question.split())
        option_tokens = set(option.split())
        option_words = option.split()

        position = 0.0
        if option_words and option_words[0] in article:
            position = article.find(option_words[0]) / max(len(article), 1)

        rows.append(
            [
                len(option_words),
                len(question.split()),
                len(question_tokens & option_tokens),
                len(option_tokens & article_tokens),
                position,
            ]
        )
    return csr_matrix(np.asarray(rows, dtype=np.float32))


def build_features(
    train_exp,
    val_exp,
    test_exp,
    save_dir=DEFAULT_PROCESSED_DIR,
    models_dir=DEFAULT_MODELS_DIR,
    max_features=5000,
):
    """Build and save OHE + cosine + lexical feature matrices."""
    save_dir = _resolve_path(save_dir)
    models_dir = _resolve_path(models_dir)
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(models_dir, exist_ok=True)

    vectorizer = CountVectorizer(binary=True, max_features=max_features, min_df=2)
    train_ohe = vectorizer.fit_transform(train_exp["combined_text"])
    val_ohe = vectorizer.transform(val_exp["combined_text"])
    test_ohe = vectorizer.transform(test_exp["combined_text"])
    joblib.dump(vectorizer, os.path.join(models_dir, "ohe_vectorizer.pkl"))

    for split_name, expanded_df, sparse_ohe in (
        ("train", train_exp, train_ohe),
        ("val", val_exp, val_ohe),
        ("test", test_exp, test_ohe),
    ):
        full_matrix = hstack(
            [
                sparse_ohe,
                _rowwise_cosine(expanded_df["article"], expanded_df["option"], vectorizer.vocabulary_),
                _lexical_features(expanded_df),
            ],
            format="csr",
        )
        save_npz(os.path.join(save_dir, f"X_{split_name}.npz"), full_matrix)
        np.save(os.path.join(save_dir, f"y_{split_name}.npy"), expanded_df["label"].to_numpy())

    return vectorizer


def load_raw_splits(raw_dir=DEFAULT_RAW_DIR):
    """Load train, validation, and test CSV files."""
    raw_dir = _resolve_path(raw_dir)
    train_df = pd.read_csv(os.path.join(raw_dir, "train.csv"))
    val_df = pd.read_csv(os.path.join(raw_dir, "val.csv"))
    test_df = pd.read_csv(os.path.join(raw_dir, "test.csv"))
    return train_df, val_df, test_df


def save_expanded_splits(train_exp, val_exp, test_exp, save_dir=DEFAULT_PROCESSED_DIR):
    """Persist expanded splits for debugging and CLI random sampling."""
    save_dir = _resolve_path(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    train_exp.to_csv(os.path.join(save_dir, "train_exp.csv"), index=False)
    val_exp.to_csv(os.path.join(save_dir, "val_exp.csv"), index=False)
    test_exp.to_csv(os.path.join(save_dir, "test_exp.csv"), index=False)


def preprocess_and_build(raw_dir=DEFAULT_RAW_DIR, save_dir=DEFAULT_PROCESSED_DIR, models_dir=DEFAULT_MODELS_DIR):
    """Run the complete raw CSV to saved features pipeline."""
    train_df, val_df, test_df = load_raw_splits(raw_dir)
    train_exp = expand_df(prepare_text_columns(train_df))
    val_exp = expand_df(prepare_text_columns(val_df))
    test_exp = expand_df(prepare_text_columns(test_df))
    save_expanded_splits(train_exp, val_exp, test_exp, save_dir)
    build_features(train_exp, val_exp, test_exp, save_dir, models_dir)
    return train_exp, val_exp, test_exp


def load_features(processed_dir=DEFAULT_PROCESSED_DIR):
    """Load saved sparse matrices and labels."""
    processed_dir = _resolve_path(processed_dir)
    X_train = load_npz(os.path.join(processed_dir, "X_train.npz"))
    X_val = load_npz(os.path.join(processed_dir, "X_val.npz"))
    X_test = load_npz(os.path.join(processed_dir, "X_test.npz"))
    y_train = np.load(os.path.join(processed_dir, "y_train.npy"))
    y_val = np.load(os.path.join(processed_dir, "y_val.npy"))
    y_test = np.load(os.path.join(processed_dir, "y_test.npy"))
    return X_train, X_val, X_test, y_train, y_val, y_test


__all__ = [
    "build_features",
    "clean_text",
    "expand_df",
    "load_features",
    "load_raw_splits",
    "prepare_text_columns",
    "preprocess_and_build",
    "save_expanded_splits",
]


In [ ]:
%%writefile src/evaluate.py
"""Evaluation helpers for classification and text generation."""

import re
import string

import numpy as np
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


def compute_metrics(y_true, y_pred, y_proba=None, n_options=4):
    """Compute binary verification metrics, plus MCQ exact match when probabilities are supplied."""
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if y_proba is not None:
        y_true = np.asarray(y_true)
        y_proba = np.asarray(y_proba)
        n_groups = len(y_true) // n_options
        correct = 0
        counted = 0
        for group_idx in range(n_groups):
            start = group_idx * n_options
            end = start + n_options
            true_group = y_true[start:end]
            proba_group = y_proba[start:end]
            if len(true_group) == n_options and true_group.sum() > 0:
                correct += int(np.argmax(proba_group) == np.argmax(true_group))
                counted += 1
        metrics["exact_match"] = float(correct / max(counted, 1))
    return metrics


def _clean_for_eval(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


def compute_generation_metrics(pred_texts, ref_texts):
    """Compute average BLEU, ROUGE-1, ROUGE-L, and METEOR for paired texts."""
    if len(pred_texts) != len(ref_texts):
        raise ValueError("pred_texts and ref_texts must have the same length")
    if not pred_texts:
        return {"bleu": 0.0, "rouge_1": 0.0, "rouge_l": 0.0, "meteor": 0.0}

    smooth = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    bleu_values = []
    rouge_1_values = []
    rouge_l_values = []
    meteor_values = []

    for pred, ref in zip(pred_texts, ref_texts):
        pred_clean = _clean_for_eval(pred)
        ref_clean = _clean_for_eval(ref)
        pred_tokens = pred_clean.split()
        ref_tokens = ref_clean.split()

        bleu_values.append(
            float(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth))
            if pred_tokens and ref_tokens
            else 0.0
        )
        rouge = scorer.score(ref_clean, pred_clean)
        rouge_1_values.append(float(rouge["rouge1"].fmeasure))
        rouge_l_values.append(float(rouge["rougeL"].fmeasure))
        meteor_values.append(float(meteor_score([ref_tokens], pred_tokens)) if pred_tokens and ref_tokens else 0.0)

    return {
        "bleu": float(np.mean(bleu_values)),
        "rouge_1": float(np.mean(rouge_1_values)),
        "rouge_l": float(np.mean(rouge_l_values)),
        "meteor": float(np.mean(meteor_values)),
    }


In [ ]:
%%writefile src/model_a_train.py
"""Train Model A answer verification classifiers."""

import os
import time
import warnings

import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

try:
    from .evaluate import compute_metrics
    from .preprocessing import DEFAULT_PROCESSED_DIR, PROJECT_ROOT, load_features
except ImportError:
    from evaluate import compute_metrics
    from preprocessing import DEFAULT_PROCESSED_DIR, PROJECT_ROOT, load_features


warnings.filterwarnings("ignore")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models", "model_a", "traditional")


def main(processed_dir=DEFAULT_PROCESSED_DIR, models_dir=MODELS_DIR):
    os.makedirs(models_dir, exist_ok=True)
    X_train, X_val, _X_test, y_train, y_val, _y_test = load_features(processed_dir)
    X_lex_train = X_train[:, -5:].toarray()
    X_lex_val = X_val[:, -5:].toarray()

    classifiers = [
        (
            "lr",
            LogisticRegression(max_iter=1000, C=1.0, solver="saga", n_jobs=-1),
            X_train,
            X_val,
        ),
        (
            "svm",
            CalibratedClassifierCV(LinearSVC(max_iter=2000), cv=3),
            X_train,
            X_val,
        ),
        (
            "rf",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1,
            ),
            X_lex_train,
            X_lex_val,
        ),
    ]

    for name, classifier, train_matrix, val_matrix in classifiers:
        start = time.time()
        classifier.fit(train_matrix, y_train)
        elapsed = time.time() - start
        predictions = classifier.predict(val_matrix)
        metrics = compute_metrics(y_val, predictions)
        print(f"{name}: acc={metrics['accuracy']:.4f}, f1={metrics['macro_f1']:.4f}, time={elapsed:.1f}s")
        joblib.dump(classifier, os.path.join(models_dir, f"{name}_model.pkl"))

    print("All Model A classifiers trained and saved.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/model_b_train.py
"""Train Model B distractor and hint scoring models."""

import os
import re
import warnings
from collections import Counter

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

try:
    from .preprocessing import DEFAULT_RAW_DIR, PROJECT_ROOT, clean_text
except ImportError:
    from preprocessing import DEFAULT_RAW_DIR, PROJECT_ROOT, clean_text


warnings.filterwarnings("ignore")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models", "model_b", "traditional")
OPTIONS = ("A", "B", "C", "D")


def extract_candidates(article, answer, max_ngram=2, limit=None):
    tokens = clean_text(article).split()
    answer_clean = clean_text(answer)
    candidates = []
    seen = set()
    for ngram_size in range(1, max_ngram + 1):
        for idx in range(len(tokens) - ngram_size + 1):
            candidate = " ".join(tokens[idx : idx + ngram_size])
            if candidate == answer_clean or len(candidate) <= 2 or candidate in seen:
                continue
            seen.add(candidate)
            candidates.append(candidate)
            if limit and len(candidates) >= limit:
                return candidates
    return candidates


def _char_match_score(candidate, answer):
    return sum(1 for a, b in zip(candidate, answer) if a == b) / max(len(answer), 1)


def _distractor_features(candidate, answer, article, vectorizer):
    candidate_vec = vectorizer.transform([candidate])
    answer_clean = clean_text(answer)
    article_clean = clean_text(article)
    answer_vec = vectorizer.transform([answer_clean])
    article_vec = vectorizer.transform([article_clean])
    article_tokens = article_clean.split()
    return [
        cosine_similarity(candidate_vec, answer_vec)[0, 0],
        cosine_similarity(candidate_vec, article_vec)[0, 0],
        article_tokens.count(candidate.split()[0]) / max(len(article_tokens), 1),
        _char_match_score(candidate, answer_clean),
        len(candidate.split()) / max(len(answer_clean.split()), 1),
    ]


def _hint_features(sentence, question, answer, position, total_sentences):
    question_tokens = set(clean_text(question).split())
    answer_tokens = set(clean_text(answer).split())
    sentence_tokens = set(clean_text(sentence).split())
    answer_overlap = len(answer_tokens & sentence_tokens) / max(len(answer_tokens), 1)
    return [
        len(question_tokens & sentence_tokens) / max(len(question_tokens), 1),
        answer_overlap,
        position / max(total_sentences, 1),
        len(sentence.split()),
    ], answer_overlap


def main(raw_dir=DEFAULT_RAW_DIR, models_dir=MODELS_DIR, sample_size=500):
    os.makedirs(models_dir, exist_ok=True)
    train_df = pd.read_csv(os.path.join(raw_dir, "train.csv"))

    vectorizer = CountVectorizer(binary=True, max_features=3000)
    vectorizer.fit(train_df["article"].apply(clean_text).tolist())
    joblib.dump(vectorizer, os.path.join(models_dir, "vectorizer_b.pkl"))

    sample_n = min(sample_size, len(train_df))
    sampled = train_df.sample(sample_n, random_state=42)

    distractor_x = []
    distractor_y = []
    for _, row in sampled.iterrows():
        correct_answer = row[row["answer"]]
        gold_distractors = [clean_text(row[option]) for option in OPTIONS if option != row["answer"]]
        for candidate in extract_candidates(row["article"], correct_answer, limit=30):
            distractor_x.append(_distractor_features(candidate, correct_answer, row["article"], vectorizer))
            distractor_y.append(int(any(candidate in gold or gold in candidate for gold in gold_distractors)))

    distractor_x = np.asarray(distractor_x, dtype=np.float32)
    distractor_y = np.asarray(distractor_y, dtype=np.int64)
    if len(set(distractor_y.tolist())) < 2:
        distractor_y[0] = 1 - distractor_y[0]
    distractor_ranker = LogisticRegression(max_iter=500)
    distractor_ranker.fit(distractor_x, distractor_y)
    joblib.dump(distractor_ranker, os.path.join(models_dir, "distractor_ranker.pkl"))
    print(f"Distractor ranker trained. Acc: {accuracy_score(distractor_y, distractor_ranker.predict(distractor_x)):.4f}")

    hint_x = []
    hint_y = []
    for _, row in sampled.iterrows():
        answer = row[row["answer"]]
        sentences = [sentence.strip() for sentence in re.split(r"[.!?]", str(row["article"])) if len(sentence.strip()) > 15]
        for pos, sentence in enumerate(sentences[:15]):
            features, answer_overlap = _hint_features(sentence, row["question"], answer, pos, len(sentences))
            hint_x.append(features)
            hint_y.append(1 if answer_overlap > 0.3 else 0)

    hint_x = np.asarray(hint_x, dtype=np.float32)
    hint_y = np.asarray(hint_y, dtype=np.int64)
    if len(set(hint_y.tolist())) < 2:
        hint_y[0] = 1 - hint_y[0]
    hint_scorer = LogisticRegression(max_iter=500)
    hint_scorer.fit(hint_x, hint_y)
    joblib.dump(hint_scorer, os.path.join(models_dir, "hint_scorer.pkl"))
    print(f"Hint scorer trained. Acc: {accuracy_score(hint_y, hint_scorer.predict(hint_x)):.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/inference.py
"""Unified inference API for answer prediction, distractors, hints, and questions."""

import os
import re
import string
import warnings
from collections import Counter

import joblib
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


warnings.filterwarnings("ignore")
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)
MODELS_A = os.path.join(PROJECT_ROOT, "models", "model_a", "traditional")
MODELS_B = os.path.join(PROJECT_ROOT, "models", "model_b", "traditional")
OPTIONS = ("A", "B", "C", "D")
_MODELS = {}


def _load_models():
    if _MODELS:
        return
    _MODELS["ohe"] = joblib.load(os.path.join(MODELS_A, "ohe_vectorizer.pkl"))
    _MODELS["lr"] = joblib.load(os.path.join(MODELS_A, "lr_model.pkl"))
    _MODELS["svm"] = joblib.load(os.path.join(MODELS_A, "svm_model.pkl"))
    _MODELS["rf"] = joblib.load(os.path.join(MODELS_A, "rf_model.pkl"))
    _MODELS["dist_vec"] = joblib.load(os.path.join(MODELS_B, "vectorizer_b.pkl"))
    _MODELS["dist_ranker"] = joblib.load(os.path.join(MODELS_B, "distractor_ranker.pkl"))
    _MODELS["hint_scorer"] = joblib.load(os.path.join(MODELS_B, "hint_scorer.pkl"))


def _clean(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


def _lexical_row(article, question, option):
    article_tokens = set(article.split())
    question_tokens = set(question.split())
    option_tokens = set(option.split())
    option_words = option.split()
    position = 0.0
    if option_words and option_words[0] in article:
        position = article.find(option_words[0]) / max(len(article), 1)
    return [
        len(option_words),
        len(question.split()),
        len(question_tokens & option_tokens),
        len(option_tokens & article_tokens),
        position,
    ]


def _answer_feature_matrix(article, question, options):
    ohe = _MODELS["ohe"]
    article_clean = _clean(article)
    question_clean = _clean(question)
    option_clean = [_clean(option) for option in options]
    combined = [f"{article_clean} [SEP] {question_clean} [SEP] {option}" for option in option_clean]

    ohe_matrix = ohe.transform(combined)
    shared_vectorizer = CountVectorizer(binary=True, vocabulary=ohe.vocabulary_)
    article_matrix = shared_vectorizer.transform([article_clean] * len(options))
    option_matrix = shared_vectorizer.transform(option_clean)
    cosine_values = [
        cosine_similarity(article_matrix[idx], option_matrix[idx])[0, 0] for idx in range(len(options))
    ]
    cosine_feature = csr_matrix(np.asarray(cosine_values, dtype=np.float32).reshape(-1, 1))
    lexical_feature = csr_matrix(
        np.asarray([_lexical_row(article_clean, question_clean, option) for option in option_clean], dtype=np.float32)
    )
    return hstack([ohe_matrix, cosine_feature, lexical_feature], format="csr"), lexical_feature.toarray()


def _positive_probability(model, matrix):
    probabilities = model.predict_proba(matrix)
    if probabilities.shape[1] == 1:
        return probabilities[:, 0]
    if hasattr(model, "classes_"):
        classes = list(model.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
    return probabilities[:, -1]


def predict_answer(article, question, options):
    """Return the predicted answer letter: A, B, C, or D."""
    if len(options) != 4:
        raise ValueError("predict_answer expects exactly 4 options")
    _load_models()
    full_matrix, lexical_matrix = _answer_feature_matrix(article, question, options)
    lr_scores = _positive_probability(_MODELS["lr"], full_matrix)
    svm_scores = _positive_probability(_MODELS["svm"], full_matrix)
    rf_scores = _positive_probability(_MODELS["rf"], lexical_matrix)
    scores = (lr_scores + svm_scores + rf_scores) / 3.0
    return OPTIONS[int(np.argmax(scores))]


def _extract_ngram_candidates(article, answer, max_ngram=2):
    tokens = _clean(article).split()
    answer_clean = _clean(answer)
    candidates = []
    seen = set()
    stop = {"the", "a", "an", "is", "was", "are", "were", "of", "in", "to", "and", "or", "it"}
    for ngram_size in range(1, max_ngram + 1):
        for idx in range(len(tokens) - ngram_size + 1):
            candidate = " ".join(tokens[idx : idx + ngram_size])
            if candidate in seen or candidate == answer_clean or len(candidate) <= 2:
                continue
            if all(token in stop for token in candidate.split()):
                continue
            if candidate in answer_clean or answer_clean in candidate:
                continue
            seen.add(candidate)
            candidates.append(candidate)
    return candidates


def _distractor_features(candidate, answer, article, vectorizer):
    candidate_vec = vectorizer.transform([candidate])
    answer_clean = _clean(answer)
    article_clean = _clean(article)
    answer_vec = vectorizer.transform([answer_clean])
    article_vec = vectorizer.transform([article_clean])
    article_tokens = article_clean.split()
    return [
        cosine_similarity(candidate_vec, answer_vec)[0, 0],
        cosine_similarity(candidate_vec, article_vec)[0, 0],
        article_tokens.count(candidate.split()[0]) / max(len(article_tokens), 1),
        sum(1 for a, b in zip(candidate, answer_clean) if a == b) / max(len(answer_clean), 1),
        len(candidate.split()) / max(len(answer_clean.split()), 1),
    ]


def generate_distractors(article, question, answer, n=3):
    """Generate n diverse distractor strings from the passage."""
    _load_models()
    vectorizer = _MODELS["dist_vec"]
    ranker = _MODELS["dist_ranker"]
    candidates = _extract_ngram_candidates(article, answer)
    if len(candidates) < n:
        stop = {"the", "a", "an", "is", "was", "are", "were", "of", "in", "to", "and", "or", "it"}
        answer_tokens = set(_clean(answer).split())
        for token, _count in Counter(_clean(article).split()).most_common(50):
            if token not in stop and token not in answer_tokens and token not in candidates and len(token) > 2:
                candidates.append(token)

    if not candidates:
        return ["Cannot be determined", "Not stated in the passage", "None of the above"][:n]

    limited_candidates = candidates[:120]
    feature_matrix = np.asarray(
        [_distractor_features(candidate, answer, article, vectorizer) for candidate in limited_candidates],
        dtype=np.float32,
    )
    scores = _positive_probability(ranker, feature_matrix)
    ranked = [limited_candidates[idx] for idx in np.argsort(scores)[::-1]]

    selected = []
    for candidate in ranked:
        if len(selected) >= n:
            break
        candidate_vec = vectorizer.transform([candidate])
        too_similar = any(
            cosine_similarity(candidate_vec, vectorizer.transform([existing]))[0, 0] > 0.8 for existing in selected
        )
        if not too_similar:
            selected.append(candidate)

    fallbacks = ["Cannot be determined", "Not stated in the passage", "None of the above", "All of the above"]
    for fallback in fallbacks:
        if len(selected) >= n:
            break
        if fallback not in selected and _clean(fallback) != _clean(answer):
            selected.append(fallback)
    return selected[:n]


def _hint_features(sentence, question, answer="", position=0, total_sentences=1):
    question_tokens = set(_clean(question).split())
    answer_tokens = set(_clean(answer).split())
    sentence_tokens = set(_clean(sentence).split())
    return [
        len(question_tokens & sentence_tokens) / max(len(question_tokens), 1),
        len(answer_tokens & sentence_tokens) / max(len(answer_tokens), 1) if answer_tokens else 0.0,
        position / max(total_sentences, 1),
        len(sentence.split()),
    ]


def get_hints(article, question, n=3):
    """Return n graduated hints from lower to higher model relevance."""
    _load_models()
    sentences = [sentence.strip() for sentence in re.split(r"[.!?]", str(article)) if len(sentence.strip()) > 10]
    if not sentences:
        return ["Refer back to the passage for more context."] * n

    features = np.asarray(
        [_hint_features(sentence, question, position=idx, total_sentences=len(sentences)) for idx, sentence in enumerate(sentences)],
        dtype=np.float32,
    )
    scores = _positive_probability(_MODELS["hint_scorer"], features)
    order = np.argsort(scores)
    if len(order) <= n:
        chosen_indices = list(order)
    else:
        positions = np.linspace(0, len(order) - 1, n).round().astype(int)
        chosen_indices = [int(order[pos]) for pos in positions]

    hints = []
    seen = set()
    for idx in chosen_indices:
        hint = sentences[idx]
        key = _clean(hint)
        if key and key not in seen:
            seen.add(key)
            hints.append(hint)
    while len(hints) < n:
        hints.append("Refer back to the passage for more context.")
    return hints[:n]


def _score_sentences(article, answer, top_k=3):
    sentences = [sentence.strip() for sentence in re.split(r"[.!?]", str(article)) if len(sentence.strip()) > 10]
    answer_tokens = set(_clean(answer).split())
    scored = []
    for sentence in sentences:
        sentence_tokens = set(_clean(sentence).split())
        overlap = len(sentence_tokens & answer_tokens) / max(len(answer_tokens), 1)
        scored.append((overlap, sentence))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _score, sentence in scored[:top_k]]


def _question_templates(sentence, answer):
    candidates = []
    answer_clean = _clean(answer)
    sentence_clean = _clean(sentence)
    if answer_clean and answer_clean in sentence_clean:
        blanked = re.sub(re.escape(answer_clean), "___", sentence_clean, count=1)
        candidates.append(("fill_blank", f'Fill in the blank: "{blanked}"'))

    capitalized = re.findall(r"\b[A-Z][a-z]{2,}\b", sentence)
    if capitalized:
        candidates.append(("who_what", f"Who or what is {capitalized[0]}?"))

    words = sentence.split()
    if len(words) >= 4:
        subject = " ".join(words[:2]).strip(string.punctuation)
        candidates.append(("generic_what", f"What can be said about {subject}?"))
    return candidates


def generate_question(article, answer):
    """Generate candidate questions for an article and answer."""
    results = []
    seen = set()
    for sentence in _score_sentences(article, answer, top_k=3):
        for template_name, question in _question_templates(sentence, answer):
            key = _clean(question)
            if key in seen:
                continue
            seen.add(key)
            results.append(
                {
                    "question": question,
                    "answer": answer,
                    "source_sentence": sentence,
                    "template": template_name,
                }
            )
    return results
